In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import numpy as onp
import jax.numpy as jnp
import jax

from msmjax.utils.benchmarking import path_input_structures
from msmjax.calculators import set_up_msm_params, create_msm

from run_timing import make_timed_eval

In [2]:
structures = onp.load(path_input_structures / "structures_5000.npz")

In [3]:
idx_structure = 0

pos = jax.device_put(structures["positions"][idx_structure])
chg = jax.device_put(structures["charges"][idx_structure])
cll = jax.device_put(structures["cells"][idx_structure])

In [4]:
params = set_up_msm_params(
    cell=cll,
    level_one_spacings=1.0,
    level_zero_cutoff=3.0,
    pbc=(True, True, True),
    cell_mode="ortho",  # TODO
    # cell_mode="triclinic",
    dynamic_cell=False,  # TODO
    # dynamic_cell=True,
)
msm_evaluation_fns = create_msm(params)

timing_fn = make_timed_eval(
    fn=msm_evaluation_fns["energy"],
    tree_type_output=True,  # TODO
)

In [5]:
timing_fn(pos, chg)

(0.0014217356199515052, Array(-178.48175, dtype=float32))

In [6]:
idx_structure = 1

pos = jax.device_put(structures["positions"][idx_structure])
chg = jax.device_put(structures["charges"][idx_structure])

timing_fn(pos, chg)

(0.0013875869400362716, Array(-146.40332, dtype=float32))

In [7]:
calc_energy = jax.jit(msm_evaluation_fns["energy"])

In [8]:
%timeit jax.tree.flatten(calc_energy(pos, chg))[0][0].block_until_ready()

1.53 ms ± 166 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [9]:
%timeit calc_energy(pos, chg).block_until_ready()

1.47 ms ± 12.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
